# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ladinski/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My Lane as an ML Task

**Selected lane:** Refresh / Content Opportunity Scoring

**Primary task type:** Ranking / Scoring

My goal is to rank existing web pages according to how urgently they deserve editorial review.

The system should help an SEO specialist decide which pages to investigate first when reviewing every page is impractical.

I will explore supervised binary classification as a supporting modeling approach. A classifier can estimate the probability that a page belongs to the declining-performance category, and those probabilities can be used to rank pages.

**Input features:** Historical search impressions, average search position, click-through rate, content age, days since last update, and other observable signals available at the decision point.

**Expected output:** A ranked review queue containing each page's predicted decline risk, review priority, and interpretable reasons for the recommendation.

**Business action:** Content specialists investigate high-priority pages and decide whether improvements are justified.

The goal is not simply to predict labels accurately, but to make better prioritization decisions than a fixed rule.

A predicted decline risk is a prioritization signal, not a direct estimate of the benefit of editing a page.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or Proxy

**Target variable:** `is_declining_label`

**Task:** Binary classification used to support page ranking.

The model will estimate whether a web page belongs to the declining-performance category.

The target has two possible values:

- **1 — Declining:** The page is classified as experiencing declining search performance.
- **0 — Not declining:** The page is not classified as declining.

In the starter dataset, `trend_direction` provides the observed performance category. We can construct the binary target as:

`is_declining_label = (trend_direction == "down").astype(int)`

### Why This Is a Proxy

Our actual business objective is to identify pages where editorial review may be valuable.

However, the dataset does not directly tell us whether editing a particular page would improve its performance.

Therefore, declining search performance is a **proxy target**, not a direct measurement of the potential benefit of content updates.

### Avoiding Data Leakage

Features must be available at the time a recommendation would be made.

For example, `trend_pct` should not be used as an input when predicting the decline label derived from it. Doing so would reveal the answer to the model.

For a future deployment, I would also ensure that the input features precede the outcome being predicted.

### Expected Model Output

The classifier will estimate a probability between 0 and 1 for each page.

Higher predicted probabilities indicate greater estimated decline risk and can be used to create a prioritized editorial review queue.

These probabilities should be interpreted as model estimates, not guarantees of future decline or improvement after editing.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success Metric

**Primary evaluation metric:** Precision@50

Since the goal is to prioritize web pages for editorial review, ranking quality matters more than overall classification accuracy.

### Why Precision@50?

Assume an editorial team has enough time to review only 50 pages.

Precision@50 measures what fraction of the 50 highest-ranked pages are actually labeled as declining.

**Formula:**

Precision@50 = Number of declining pages in the top 50 / 50

A higher Precision@50 means fewer review slots are allocated to pages that are not labeled as declining.

### Baseline Comparison

In the Week 1 starter pipeline, using client-holdout validation:

- Hand-written rule: Precision@50 = 0.240
- Random Forest: Precision@50 = 0.740

The Random Forest identified 37 declining pages among its top 50 recommendations, compared with 12 for the baseline.

This suggests that learned ranking may outperform fixed rules on the starter dataset.

### Secondary Metrics

- **Precision@20:** Measures ranking quality when editorial capacity is especially limited.
- **Recall@50:** Measures what fraction of all declining pages in the evaluation set appear in the top 50.
- **PR-AUC (Average Precision):** Evaluates the model's ability to identify declining pages across different decision thresholds.

### Evaluation Strategy

I will use client-holdout validation to avoid evaluating a model on pages from clients it encountered during training.

I will also compare the model against transparent hand-written rules and inspect errors.

If the project later has time-ordered outcomes, I would use temporal validation to test whether the model generalizes to future periods.

### Business Success

The system should help editors identify more relevant pages within a limited review budget.

However, high Precision@50 measures how well the model identifies the decline proxy, not whether refreshing those pages will improve traffic.

Actual business impact would require observing editorial interventions and subsequent outcomes, ideally through a controlled evaluation.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Unit of Analysis - One Web Page

**Unit of analysis:** One web page represented by one row in the dataset.

Each row contains observable information about a page, including its search performance, content characteristics, and update history.

**Input features:**
- `impressions_90d` — Search impressions over 90 days.
- `avg_position` — Average search ranking position.
- `ctr` — Click-through rate.
- `content_age_days` — Age of the content.
- `days_since_last_update` — Time since the page was last updated.
- `word_count` — Length of the content.

**Target:** `is_declining_label`, derived from `trend_direction`.

The model will learn relationships between the input features and the decline label, then assign scores that can be used to rank pages for editorial review.

The following code loads the starter dataset, constructs the binary target, and displays a sample of actual observations.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import subprocess
import pandas as pd

# Load the public FlyRank starter repository
repo_dir = "/content/flyrank-ml-internship"

if not os.path.exists(repo_dir):
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/Ladinski/flyrank-ml-internship.git",
        repo_dir
    ], check=True)

# Load the anonymized starter data
csv_path = f"{repo_dir}/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(csv_path)

# Construct our binary classification target
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

features = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

# Display actual features and target
display(df[features + ["is_declining_label"]].head(10))

print(f"\nTotal observations: {len(df):,}")
print("Unit of analysis: one web page")

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget percentages:")
print(
    df["is_declining_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)



,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update,word_count,is_declining_label
0,3803,10.6,0.76,187,20,3221.0,1
1,15320,20.3,0.05,445,25,2481.0,1
2,12581,36.5,0.09,141,20,3515.0,1
3,11751,6.2,0.49,463,22,NaN,0
4,19140,44.0,0.13,263,14,2803.0,1
5,3970,8.5,0.03,147,20,3080.0,1
6,20,7.0,0.00,90,20,3059.0,1
7,1724,21.2,0.06,445,22,NaN,0
8,32574,46.0,0.09,90,20,3807.0,1
9,1240,4.9,0.16,257,104,NaN,1



Total observations: 30,000
Unit of analysis: one web page

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target percentages:
is_declining_label
1    54.21
0    45.79
Name: proportion, dtype: float64


## 5. Why Machine Learning Instead of a Fixed Rule?

A fixed rule might prioritize pages that have not been updated for at least 180 days and receive more than 500 search impressions.

Although this approach is simple and interpretable, it relies on manually selected thresholds and cannot easily capture complex relationships between multiple features.

### Advantages of Machine Learning

**1. Combining multiple signals**

Machine learning can consider search impressions, CTR, average position, content age, update recency, and content length simultaneously.

**2. Learning patterns from data**

Rather than relying entirely on manually defined thresholds, a model can identify statistical relationships associated with declining performance.

**3. Better prioritization**

In the starter pipeline's client-holdout evaluation, the Random Forest achieved Precision@50 of 0.740 compared with 0.240 for the baseline rule.

This provides preliminary evidence that learned ranking may identify more declining pages within a limited editorial review budget.

**4. Adapting to new data**

Models can be retrained and evaluated as additional observations become available, although retraining does not guarantee improved performance.

### Why a Fixed Rule Still Matters

Fixed rules are transparent, inexpensive to implement, and easy to maintain.

They provide an important baseline. A machine learning system should only be preferred when it demonstrates meaningful improvements on unseen data relative to its additional complexity.

### Limitations

The current target identifies pages labeled as declining, not pages that would necessarily benefit from editorial updates.

Missing values, data leakage, differences between clients, and changes in search behavior could affect model reliability.

I will use appropriate preprocessing, client-holdout validation, and transparent comparisons to evaluate whether machine learning offers a meaningful advantage.

**Conclusion:** Machine learning is worth investigating because it can combine multiple signals and potentially improve review prioritization, but its value must be demonstrated through reliable evaluation and business outcomes.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.